# 2. Multiclass preselection


    Train a three-class classifier on S, B, and NI, then freeze a cut on its S
    output. The aim is to improve the physical S/NI yield ratio from 0.01 to
    approximately 0.1, without using the eventual inference sample to choose the
    cut. Run notebook 1 first. A GPU is recommended in production mode.

### Runtime and persistent run

In Colab, choose **Runtime → Change runtime type → GPU** for notebooks 2–3.
PyTorch uses the GPU for training; the JAX likelihood fits run on the CPU.
Setup removes preinstalled JAX CUDA plugins that are incompatible with the pinned
JAX version, then checks the JAX runtime before training. If an earlier session
reported a JAX plugin error, restart the runtime once before rerunning setup.
Keep the same run name: saved samples and trained networks are reused.
The source checkout lives in the temporary runtime; **datasets, checkpoints,
workspace files, and results live in Google Drive**. Use the same `RUN_NAME`
and `MODE` in every notebook. Set `MODE = "smoke"` for a short workflow check;
the production default generates five million events **per sample**.
Use `CONFIG_OVERRIDES` to change a computational budget without editing source,
for example `{"quadrature_per_process": 500_000, "epochs": 150}`. Use the same
overrides in all five notebooks, and choose a new `RUN_NAME` when changing them.

This repository may be private. Grant Colab access when opening it from GitHub.
For the runtime download, either save a GitHub fine-grained token with **Contents:
read** access to this repository as the Colab secret `GITHUB_TOKEN`, or enter it
in the hidden prompt. The token is used only in a Python HTTP header and is not
written into files, Git remotes, shell arguments, or notebook output. Alternatively,
upload a repository ZIP to `/content/poodemo_source.zip`; no token is then needed.
An existing `/content/poodemo` checkout is reused. Delete that checkout to fetch
new source after updating the repository; your Drive run is separate.
For this NI-only revision, use the new default `RUN_NAME = "demo-ni-v1"`.
If you ran an earlier version, start a fresh Colab runtime (or refresh the
temporary source checkout and restart the runtime) before proceeding. Existing
run manifests from the earlier model are incompatible and are not reused.

In [ ]:
import os
import sys
from pathlib import Path

RUN_NAME = "demo-ni-v1"
MODE = os.environ.get("POODEMO_MODE", "production")  # production or smoke
CONFIG_OVERRIDES = {}  # e.g. {"quadrature_per_process": 500_000, "epochs": 150}

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # JAX likelihood fits use the validated CPU runtime. This does not change
    # PyTorch's CUDA visibility or GPU training.
    os.environ["JAX_PLATFORMS"] = "cpu"
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    ROOT = Path(os.environ.get("POODEMO_ROOT", f"/content/drive/MyDrive/poodemo/runs/{RUN_NAME}"))
    REPO_DIR = Path("/content/poodemo")
    if not (REPO_DIR / "pyproject.toml").exists():
        import getpass
        import io
        import shutil
        import tarfile
        import tempfile
        import urllib.error
        import urllib.request
        import zipfile

        uploaded_zip = Path("/content/poodemo_source.zip")
        if uploaded_zip.exists():
            archive_bytes = uploaded_zip.read_bytes()
            archive_kind = "zip"
        else:
            archive_url = "https://api.github.com/repos/rafaellopesdesa/poodemo/tarball/main"

            def fetch_source(token=None):
                headers = {"Accept": "application/vnd.github+json", "User-Agent": "poodemo-colab"}
                if token:
                    headers["Authorization"] = "Bearer " + token
                request = urllib.request.Request(archive_url, headers=headers)
                with urllib.request.urlopen(request, timeout=180) as response:
                    return response.read()

            try:
                archive_bytes = fetch_source()
            except urllib.error.HTTPError as error:
                if error.code not in (401, 403, 404):
                    raise RuntimeError(f"Repository download failed (HTTP {error.code}).") from None
                token = None
                try:
                    from google.colab import userdata
                    token = userdata.get("GITHUB_TOKEN")
                except Exception:
                    pass
                if not token:
                    token = getpass.getpass("GitHub token with read access to poodemo: ").strip()
                if not token:
                    raise RuntimeError("Upload /content/poodemo_source.zip or provide read access.")
                try:
                    archive_bytes = fetch_source(token)
                except urllib.error.HTTPError as auth_error:
                    raise RuntimeError(f"Repository download failed (HTTP {auth_error.code}); check token access.") from None
                finally:
                    token = None
            archive_kind = "tar"

        # Validate archive paths and reject links before extracting the source.
        with tempfile.TemporaryDirectory(prefix="poodemo-source-") as temp:
            staging = Path(temp).resolve()

            def safe_destination(name):
                destination = (staging / name).resolve()
                if not destination.is_relative_to(staging):
                    raise RuntimeError("Unsafe path in source archive.")
                return destination

            if archive_kind == "zip":
                with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
                    for entry in archive.infolist():
                        safe_destination(entry.filename)
                        if (entry.external_attr >> 16) & 0o170000 == 0o120000:
                            raise RuntimeError("Symbolic links are not supported in the source ZIP.")
                    archive.extractall(staging)
            else:
                with tarfile.open(fileobj=io.BytesIO(archive_bytes), mode="r:gz") as archive:
                    for entry in archive.getmembers():
                        safe_destination(entry.name)
                        if not (entry.isfile() or entry.isdir()):
                            raise RuntimeError("Links and special files are not supported in the source archive.")
                    archive.extractall(staging)
            candidates = [p.parent for p in staging.rglob("pyproject.toml") if (p.parent / "poodemo").is_dir()]
            if len(candidates) != 1:
                raise RuntimeError("The uploaded archive must contain one poodemo repository checkout.")
            shutil.copytree(candidates[0], REPO_DIR, dirs_exist_ok=True)
        del archive_bytes

    import subprocess
    if os.environ.get("POODEMO_SKIP_INSTALL") != "1":
        # JAX discovers plugins even with JAX_PLATFORMS=cpu. Colab can retain
        # newer CUDA plugins when pip installs an older jax/jaxlib pair.
        subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-y", "-q",
                               "jax-cuda12-plugin", "jax-cuda12-pjrt",
                               "jax-cuda13-plugin", "jax-cuda13-pjrt"])
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements-colab.txt")])
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)])

        # Check the live notebook kernel before any expensive network training.
        import jax
        import jaxlib
        if jax.__version__ != "0.5.3" or jaxlib.__version__ != "0.5.3":
            raise RuntimeError("JAX packages are stale in this kernel. Restart the runtime and rerun setup.")
        jax.config.update("jax_platforms", "cpu")
        jax.config.update("jax_enable_x64", True)
        jax_devices = jax.devices()
        if any(device.platform != "cpu" for device in jax_devices):
            raise RuntimeError("Expected the CPU JAX backend for likelihood fits. Restart the runtime and rerun setup.")
        print(f"JAX {jax.__version__} / jaxlib {jaxlib.__version__}: {jax_devices}")

    # A new editable install is not activated in an already-running kernel.
    # Expose the inner package directly, including after a failed import cached
    # the outer /content/poodemo checkout as a namespace package.
    import importlib
    source_root = str(REPO_DIR.resolve())
    if source_root in sys.path:
        sys.path.remove(source_root)
    sys.path.insert(0, source_root)
    importlib.invalidate_caches()
    cached_package = sys.modules.get("poodemo")
    cached_file = getattr(cached_package, "__file__", None)
    expected_init = REPO_DIR / "poodemo" / "__init__.py"
    if cached_package is not None and (cached_file is None or Path(cached_file).resolve() != expected_init.resolve()):
        for module_name in list(sys.modules):
            if module_name == "poodemo" or module_name.startswith("poodemo."):
                del sys.modules[module_name]
else:
    # Locally: pip install -r requirements-colab.txt && pip install -e .
    ROOT = Path(os.environ.get("POODEMO_ROOT", str(Path.home() / "poodemo-runs" / RUN_NAME)))
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "poodemo" / "pipeline.py").exists():
            sys.path.insert(0, str(candidate))
            break

ROOT.mkdir(parents=True, exist_ok=True)
print(f"Run directory: {ROOT}")
print(f"Mode: {MODE}")

In [2]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from poodemo.pipeline import create_run

run = create_run(ROOT, mode=MODE, overrides=CONFIG_OVERRIDES)
plt.rcParams.update({"figure.figsize": (7, 4.5), "axes.grid": True, "grid.alpha": 0.2})
display(pd.Series(run.config, name="configuration"))

,configuration
schema_version,3
mode,production
seed,20261006
toolkit_commit,fc09848fc6540fd32310faebbe9db6eea7ecd17b
n_per_sample,5000000
generation_chunk,250000
preselection_train_cap,1000000
ratio_train_cap,2000000
epochs,100
patience,20


### Why this is a separate classifier

The preselection network uses multiclass cross entropy and the toolkit's
neural-network implementation. Its output is a selection variable,
not the final likelihood ratio or optimal score. Class-balanced training
avoids allowing the dominant NI yield alone to determine the classifier.

A threshold \(c\) is chosen on held-out calibration events to satisfy
\[
\frac{\lambda_S\epsilon_S(c)}
     {\lambda_{NI}\epsilon_{NI}(c)}\simeq0.1.
\]
Within the available threshold choices, preserving signal efficiency
matters. The target is approximate because efficiencies are estimated
with finite samples. The same frozen cut is applied to every process,
every nuisance variation, and all subsequent parameter hypotheses.

In [3]:
from poodemo.pipeline import run_preselection

selection = run_preselection(run)
display(selection)

INFO:lightning_fabric.utilities.seed:Seed set to 20261006
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.13/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/usr/local/lib/python3.13/dist-packages/pytorch_lightning/trainer/connectors/data_connector.py:433: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.
/usr/local/lib/python3.13/dist-packages/pytorch_lightning/utilities/data.py:123: Your `IterableDataset` has `__l

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

{'threshold': 0.3213997738705873,
 'target_ratio': 0.1,
 'achieved_ratio': 0.10029665686994857,
 'efficiencies': {'S': 0.873624, 'NI': 0.087104},
 'selected_counts': {'S': 218406, 'NI': 21776},
 'calibration_counts': {'S': 250000, 'NI': 250000},
 'rule': 'signal_softmax >= threshold',
 'signal_class_index': 0,
 'heldout_s_over_ni': 0.10101923699921325,
 'selection':      sample        split  generated  accepted  efficiency  expected_yield
 0         S        train    2000000   1747374    0.873687       87.368700
 1         S   validation     500000    436694    0.873388       87.338800
 2         S  calibration     250000    218245    0.872980       87.298000
 3         S  integration     750000    654837    0.873116       87.311600
 4       SBI        train    2000000    475534    0.237767      195.031109
 5       SBI   validation     500000    118729    0.237458      194.777649
 6       SBI  calibration     250000     59164    0.236656      194.119799
 7       SBI  integration     75

### Selection is part of the statistical experiment

A cut changes both rates and normalized shapes. For process \(j\),
\[
\lambda_j^{\mathrm{sel}}=\lambda_j\epsilon_j,\qquad
p_j^{\mathrm{sel}}(x)
=\frac{\mathbf1_{C}(x)p_j(x)}{\epsilon_j}.
\]
Nuisance variations generally change \(\epsilon_j\). We retain those
rate effects; renormalizing every selected variation to the nominal
rate would define a different systematic model.

Preselection training, selection tuning, density-ratio training, ratio
calibration, and final integration use separate sample roles. Only the
four density-ratio/integration partitions are materialized as selected
arrays. The learned cut is held fixed during every likelihood fit.

In [4]:
print("Selection outputs are saved under:", ROOT)
for key, value in selection.items():
    if isinstance(value, pd.DataFrame):
        print(key)
        display(value)

Selection outputs are saved under: /content/drive/MyDrive/poodemo/runs/demo-ni-v1
selection


,sample,split,generated,accepted,efficiency,expected_yield
0,S,train,2000000,1747374,0.873687,87.368700
1,S,validation,500000,436694,0.873388,87.338800
2,S,calibration,250000,218245,0.872980,87.298000
3,S,integration,750000,654837,0.873116,87.311600
4,SBI,train,2000000,475534,0.237767,195.031109
5,SBI,validation,500000,118729,0.237458,194.777649
6,SBI,calibration,250000,59164,0.236656,194.119799
7,SBI,integration,750000,178121,0.237495,194.807725
8,B,train,2000000,565596,0.282798,282.798000
9,B,validation,500000,141022,0.282044,282.044000


### What to inspect

Compare the achieved S/NI ratio with the target and inspect the retained
signal efficiency. A smoke run checks interfaces, not attainable
classifier performance. The next notebook uses the selected S sample as
the common density-ratio reference and retains all selected process
yields explicitly.

Next: **03_unbinned_nsbi.ipynb**.